# **Dataset Cleaning & Preprocessing**
### Ciaran O' Toole - C00297672
##### A portfolio produced as part of the Data Science & Machine Learning 1 Module in SE603, Bachelor of Science (Honours) Software Development at SETU

This notebook focuses on reviewing, cleaning, and validating the dataset to make it reliable and consistent and prepared for model training. I use what was initially collected and merged from TMDB and IMDb API movie records processed in `Datasetcreation.ipynb` to create `movies.csv`.

### **Step 1 - Load the Current Dataset & Do Initial Data Inspection**

First, I load the raw `movies.csv` file created during the data collection phase and perform an initial diagnostic check to identify missing values, duplicate records, invalid data types, and zero-value entries in the budget and revenue columns.

In [7]:
import pandas as pand
import numpy as np

#Load movies.csv dataset
movies = pand.read_csv("movies.csv")

print("Missing Values")
print(movies.isna().sum())
print("\nEmpty Financial Values")
print("1. Empty Budget:", (movies["budget"] == 0).sum())
print("2. Empty Revenue:", (movies["revenue"] == 0).sum())
print("\nDuplicate Values")
print("1. Duplicate IDs:", movies.duplicated(subset=["imdbID"]).sum())
print("2. duplicate rows:", movies.duplicated().sum())

Missing Values
imdbID            3
title             0
release_date      2
genres            1
budget            0
revenue           0
averageRating    16
numVotes         16
dtype: int64

Empty Financial Values
1. Empty Budget: 249
2. Empty Revenue: 226

Duplicate Values
1. Duplicate IDs: 21
2. duplicate rows: 19


### **Step 2 - Handle Duplicates & Missing Identifiers**

TMDB's `/movie/popular` endpoint which I use, updates dynamically, this can sometimes cause duplicate entries to appear. Any records without a valid imdbID, title, or release_date cannot be reliably analysed. Here, I drop rows missing core identifiers and remove duplicate entries.

In [11]:
# Check how many rows currently exist
print("Starting number of movies:", len(movies))

#Remove rows that are missing an imdbID, release_date or title fields
movies = movies.dropna(subset=["imdbID", "title", "release_date"])
print("Movies left after removal:", len(movies))
# Remove duplicate movies sharing the same ID
movies = movies.drop_duplicates(subset=["imdbID"])
print("Movies left after removing duplicates:", len(movies))
movies = movies.reset_index(drop=True)
movies.to_csv("movies.csv", index=False)

Starting number of movies: 1177
Movies left after removal: 1177
Movies left after removing duplicates: 1177


### **Step 3 - Clean Missing Ratings, Genres, and Financial Values**

During dataset creation, movies without a match generated empty values for averageRating and numVotes when being pulled from the API, however those fields are invalid so the movie entry they belong to needs to be removed to maintain accuracy. The next call to action was to mark missing genres as Unknown, in order to prevent the models from being distorted. Another thing to note is that TMDB uses 0 as a placeholder for unreported budgets and revenues, I convert those zeroes to empty values and remove them so they do not distort the accuracy of the models.

In [14]:
# look for rows that are missing ratings
missedrates = movies["averageRating"].isna().sum()
print("Movies missing IMDb ratings:", missedrates)

# remove those rows
movies = movies.dropna(subset=["averageRating"])
movies = movies.dropna(subset=["numVotes"])
# replace empty/missing genres
movies["genres"] = movies["genres"].fillna("Unknown")
movies["genres"] = movies["genres"].replace("", "Unknown")
# Replace 0 figures so they are not treated as $0
movies["budget"] = movies["budget"].replace(0, np.nan)
movies["revenue"] = movies["revenue"].replace(0, np.nan)
#drop any entry with empty fields in revenue or budget
movies = movies.dropna(subset=["budget"])
movies = movies.dropna(subset=["revenue"])
#reorder after loss of entries
movies = movies.reset_index(drop=True)
#update csv
movies.to_csv("movies.csv", index=False)
print("Movies left after step 3:", len(movies))


Movies missing IMDb ratings: 0
Movies left after step 3: 904


### **Step 4 - Standardise Release Dates to Ireland Date Time Formatting**

A quick thing, just for down the line for neatness, I refactor how dates are displayed and stored for the dataset, so they will present better on the website.

In [17]:
#converts their release date into a date time object
movies["release_date"] = pand.to_datetime(movies["release_date"], dayfirst=True, format="mixed", errors="coerce")
#get rid of any rows that failed the above conversion
movies = movies.dropna(subset=["release_date"])
#convert to irish standard of dates
movies["release_date"] = movies["release_date"].dt.strftime("%d/%m/%Y")
print("Movies left after Date Time refactor:", len(movies))

#re-adjust row numbers and save to movies.csv
movies = movies.reset_index(drop=True)
movies.to_csv("movies.csv", index=False)

Movies left after Date Time refactor: 904


### **Step 5 - Standardise The Votes Numerical Data**
After cleaning out invalid vote counts revenue and budget, this means I can now transform them from floating-point numbers back into whole integers.

In [18]:
#convert to integers
movies["numVotes"] = movies["numVotes"].astype("int64")
movies["budget"] = movies["budget"].astype("int64")
movies["revenue"] = movies["revenue"].astype("int64")

#update accordingly to the csv
movies.to_csv("movies.csv", index=False)
print("Data types updated:")
print(movies.dtypes)

Data types updated:
imdbID               str
title                str
release_date         str
genres               str
budget             int64
revenue            int64
averageRating    float64
numVotes           int64
dtype: object


### **Step 6 - Check for Invalidating Outliers**
Here i check to see what the smallest values for both revenue and budget are, as sometimes values in TMDb are not recorded properly and have low values such as 1, 2 or 100 which does not accurately represent those movies. This is a huge issue as a movie with a budget of 1 and a revenue of 100,000,000 can throw our models off massively.

In [21]:
# Check for fake placeholder values
print("Smallest 10 budgets:")
display(movies[["title", "release_date", "budget", "revenue"]].nsmallest(10, "budget"))

print("Smallest 10 revenues:")
display(movies[["title", "release_date", "budget", "revenue"]].nsmallest(10, "revenue"))

Smallest 10 budgets:


,title,release_date,budget,revenue
813,East of Eden,10/04/1955,2,5000000
123,Swapped,01/05/2026,7,7
791,Terrifier,25/01/2018,35000,421798
770,Godzilla,03/11/1954,60000,2250000
818,It Ends,21/08/2026,80000,1264397
726,Ikiru,09/10/1952,85000,55240
436,Chungking Express,14/07/1994,160000,1000000
648,Terrifier 2,06/10/2022,250000,15815602
844,Rashomon,26/08/1950,250000,117668
458,Halloween,25/10/1978,325000,70260597


Smallest 10 revenues:


,title,release_date,budget,revenue
123,Swapped,01/05/2026,7,7
726,Ikiru,09/10/1952,85000,55240
83,Mouse,06/11/2026,3000000,104112
844,Rashomon,26/08/1950,250000,117668
768,Red Notice,04/11/2021,160000000,178143
95,Paper Tiger,13/11/2026,15000000,215009
899,Hot Spot,21/08/2026,6000000,256350
687,Subservience,15/08/2024,4400000,262793
791,Terrifier,25/01/2018,35000,421798
322,Frankenstein,17/10/2025,120000000,480678


Now that I have identified that there are a few placeholder outliers, I remove those from the dataset and update the csv file accordingly.

In [23]:
#remove any movie with values for these fields that are not over 5K
movies = movies[movies["budget"] >= 5000]
movies = movies[movies["revenue"] >= 5000]
# update the dataset
movies = movies.reset_index(drop=True)
movies.to_csv("movies.csv", index=False)

print("Movies count:", len(movies))

Movies count: 902


### **Summary of DataSet Cleaning & Moving Forward**

I started this cleaning process with 1,200 movie records from `movies.csv`, I then refined it into a verified dataset of 902 high-integrity entries. I went through and removed any duplicate records and rows missing core identifiers (`imdbID`, `title`, `release_date`) or any entry lacking an IMDb ratings were also removed. Any movie with missing genres had those labelled as Unknown, and zero-value or unrealistic placeholder financial figures (under $5,000) were filtered out. This in all was to make sure these invalid fields and outliers did not distort model accuracy during training. I also modified release dates to follow Irish date formatting DD/MM/YYYY and vote counts, budgets, and revenues were converted into whole integers.

With `movies.csv` now completely free of missing or invalid values, the dataset is ready for the first model to be pre-processed and trained, where I will see how accurately the model can guess a movies revenue based on its budget. This will then be good, as the end goal is to put all the models against each other to see which is more accurate.